# 02 — PPI Resilient Infrastructure Cleaning

**What this notebook does:** Builds a country-year outcome variable measuring private investment in resilient infrastructure from the full World Bank PPI dataset across all sectors (not just electricity). The output is a cleaned, aggregated panel suitable for merging with covariates in the main analysis.

**Why we expanded beyond electricity only:** The electricity-only dataset in `01_ppi_cleaning.ipynb` was designed for the green share analysis, where technology classification required knowing the generation type. For the resilient infrastructure framing, the relevant unit is infrastructure investment broadly — transport, water, telecom, and energy together. Expanding to all sectors increases the number of projects per country-year, reduces the share of zeros in the outcome variable, and better matches standard definitions of resilient infrastructure used in the development finance literature.

**Date created:** 2026-06-24

In [ ]:
import pandas as pd
import numpy as np

print('pandas:', pd.__version__)
print('numpy: ', np.__version__)

## Load Raw Data

Load the full PPI dataset across all sectors — energy, transport, water, and telecom — from the raw Excel file. Unlike `01_ppi_cleaning.ipynb` which restricted to electricity, this file covers every sector in the World Bank PPI database for 2010–2024.

In [ ]:
df = pd.read_excel(
    '../../data/raw/PPI_resilient_all.xlsx',
    sheet_name='CustomQuery'
)

print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nFirst 3 rows:')
df.head(3)

## 1.2 Load Full PPI Dataset
We load the full World Bank PPI dataset covering all infrastructure
sectors (Energy, Transport, Water, ICT, Municipal Solid Waste)
from 2000-2024 across 130+ low and middle income countries.

In [ ]:
df = pd.read_excel(
    '../../data/raw/PPI_resilient_all.xlsx',
    sheet_name='CustomQuery'
)

print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nFirst 3 rows:')
df.head(3)

## 1.3 Explore Sectors and Coverage

Before filtering or cleaning, we inspect what sectors and subsectors are present in the full dataset, the year range covered, and how many countries are represented. This confirms the dataset matches expectations and helps identify what filtering decisions to make next.

In [ ]:
print('=== Primary sector ===')
print(df['Primary sector'].value_counts())

print('\n=== Subsector ===')
print(df['Subsector'].value_counts())

print('\n=== Year range ===')
print('Min:', df['Financial closure year'].min())
print('Max:', df['Financial closure year'].max())

print('\n=== Unique countries ===')
print(df['Country'].nunique())

## 2. Resilient vs Non-Resilient Classification

We create a binary `is_resilient` column (1 = resilient, 0 = non-resilient, NaN = ambiguous) using two different classification columns depending on the sector:

**Electricity projects** (`Subsector == 'Electricity'`) are classified by the `Technology` column, since the generation technology determines resilience — renewable generation (solar, wind, small hydro, geothermal, biomass, biogas) is classified resilient; fossil fuels (coal, gas, diesel, steam) are non-resilient. This mirrors the `is_green` logic from `01_ppi_cleaning.ipynb`.

**All other sectors** are classified by the `Subsector` column, since there is no Technology field with meaningful content outside electricity:
- **Resilient (1):** Water and sewerage infrastructure — these provide essential services with strong climate adaptation value.
- **Non-resilient (0):** Transport (roads, railways, airports, ports), ICT, natural gas distribution, and waste collection — either fossil-dependent or not primarily climate-adaptive.
- **Ambiguous (NaN):** Multi-sector hybrid subsectors (e.g. `Electricity, ICT`) and rare categories with too few rows to classify confidently.

The two-step approach means electricity resilience is technology-driven while other sectors are subsector-driven, which reflects the different information available in each part of the dataset.

In [ ]:
# STEP 1 — start with NaN for all rows
df['is_resilient'] = float('nan')

# STEP 2 — classify electricity projects by Technology column
elec = df['Subsector'] == 'Electricity'

resilient_tech = {
    'Solar, PV', 'Solar, CSP', 'Solar, CPV',
    'Wind',
    'Hydro, Small (<50MW)',
    'Geothermal', 'Biomass', 'Biogas',
    'Solar, PV, N/A', 'Wind, N/A',
    'Wind, Solar, PV', 'Solar, PV, Wind', 'Solar, PV, Solar, PV',
    'Wind, Not Applicable', 'Solar, PV, Not Applicable',
}

non_resilient_tech = {
    'Coal', 'Natural Gas', 'Diesel', 'Steam',
    'Natural Gas, Diesel', 'Natural Gas, Steam',
    'Natural Gas, Other', 'Diesel, Natural Gas',
}

df.loc[elec & df['Technology'].isin(resilient_tech),     'is_resilient'] = 1
df.loc[elec & df['Technology'].isin(non_resilient_tech), 'is_resilient'] = 0
# ambiguous electricity rows (Hydro Large, Waste, hybrids) stay NaN

# STEP 3 — classify non-electricity projects by Subsector column
non_elec = ~elec

resilient_subsector = {
    'Water and sewerage', 'Treatment plant',
    'Water Utility', 'Treatment/ Disposal',
}

non_resilient_subsector = {
    'Roads', 'Railways', 'Airports', 'Ports',
    'ICT', 'Natural Gas',
    'Collection and Transport', 'Integrated MSW',
}

df.loc[non_elec & df['Subsector'].isin(resilient_subsector),     'is_resilient'] = 1
df.loc[non_elec & df['Subsector'].isin(non_resilient_subsector), 'is_resilient'] = 0
# ambiguous non-electricity subsectors (hybrids, rare categories) stay NaN

# Summary
total = len(df)
counts = df['is_resilient'].value_counts(dropna=False)

print('is_resilient value_counts (including NaN):')
print(counts)

print('\nBreakdown:')
resilient_n     = (df['is_resilient'] == 1).sum()
non_resilient_n = (df['is_resilient'] == 0).sum()
ambiguous_n     = df['is_resilient'].isna().sum()

print(f'  Resilient     (1):  {resilient_n:>5}  ({resilient_n/total*100:.1f}%)')
print(f'  Non-resilient (0):  {non_resilient_n:>5}  ({non_resilient_n/total*100:.1f}%)')
print(f'  Ambiguous   (NaN):  {ambiguous_n:>5}  ({ambiguous_n/total*100:.1f}%)')
print(f'  Total:              {total:>5}')

## 2.1 Update: More Generous Resilient Classification

After reviewing the ambiguous rows, we reclassify several categories rather than dropping them:

- **Hydro, Large (>50MW) → Resilient:** Large hydro is carbon-free generation; while ecologically contested, it provides climate-resilient baseload power and fits the infrastructure investment framing.
- **Waste → Resilient:** Waste-to-energy contributes to waste management resilience and reduces landfill pressure, justifying inclusion.
- **E-Vehicle Charging Station → Resilient:** Supports clean transport transition.
- **Treatment plant, Water Utility / Electricity, Water Utility → Resilient:** Water-related infrastructure.
- **Electricity, ICT / Electricity, Natural Gas / Ports, Railways → Non-Resilient:** Multi-sector hybrids that include fossil or non-adaptive components; classified conservatively.
- **Not Applicable / Other / 4 hybrid noise rows → NaN:** No classifiable signal; excluded from analysis.

In [ ]:
elec = df['Subsector'] == 'Electricity'

# --- Reclassify NaN → Resilient (1) ---
# Electricity: Hydro Large and Waste
df.loc[elec & (df['Technology'] == 'Hydro, Large (>50MW)'), 'is_resilient'] = 1
df.loc[elec & (df['Technology'] == 'Waste'),                'is_resilient'] = 1

# Non-electricity: water-adjacent and clean transport
df.loc[df['Subsector'] == 'E-Vehicle Charging Station',       'is_resilient'] = 1
df.loc[df['Subsector'] == 'Treatment plant, Water Utility',   'is_resilient'] = 1
df.loc[df['Subsector'] == 'Electricity, Water Utility',       'is_resilient'] = 1

# --- Reclassify NaN → Non-Resilient (0) ---
df.loc[df['Subsector'] == 'Electricity, ICT',         'is_resilient'] = 0
df.loc[df['Subsector'] == 'Electricity, Natural Gas', 'is_resilient'] = 0
df.loc[df['Subsector'] == 'Ports, Railways',          'is_resilient'] = 0

# NaN rows kept as-is: Technology 'Not Applicable', 'Other',
# and the 4 hybrid noise rows (Wind, Coal, N/A etc.)

# Summary
total = len(df)
print('is_resilient value_counts (including NaN):')
print(df['is_resilient'].value_counts(dropna=False))

print('\nBreakdown:')
resilient_n     = (df['is_resilient'] == 1).sum()
non_resilient_n = (df['is_resilient'] == 0).sum()
ambiguous_n     = df['is_resilient'].isna().sum()

print(f'  Resilient     (1):  {resilient_n:>5}  ({resilient_n/total*100:.1f}%)')
print(f'  Non-resilient (0):  {non_resilient_n:>5}  ({non_resilient_n/total*100:.1f}%)')
print(f'  Ambiguous   (NaN):  {ambiguous_n:>5}  ({ambiguous_n/total*100:.1f}%)')
print(f'  Total:              {total:>5}')

## 3. Clean TotalInvestment Column

`TotalInvestment` is stored as a mixed-type column containing numeric values alongside placeholder strings like `"Not Available"`. We convert it to numeric before aggregation, coercing non-numeric strings to NaN. We inspect what was coerced to confirm no real data is being lost.

In [ ]:
# Identify non-numeric values before conversion
non_numeric_mask = (
    pd.to_numeric(df['TotalInvestment'], errors='coerce').isna()
    & df['TotalInvestment'].notna()
)

print('5 most common non-numeric values in TotalInvestment:')
print(df.loc[non_numeric_mask, 'TotalInvestment'].value_counts().head(5))

# Convert
before_na = df['TotalInvestment'].isna().sum()
df['TotalInvestment'] = pd.to_numeric(df['TotalInvestment'], errors='coerce')
after_na = df['TotalInvestment'].isna().sum()

new_na = after_na - before_na
pct = new_na / len(df) * 100
print(f'\nRows coerced to NaN: {new_na} ({pct:.1f}% of {len(df)} total rows)')
print(f'Already NaN before conversion: {before_na}')
print(f'Total NaN after conversion:    {after_na}')

## 4. Aggregate to Country-Year Panel

We aggregate the project-level data to a country × financial-closure-year panel. Two data quality notes before aggregating:

- **Ambiguous rows dropped:** 596 rows where `is_resilient` is NaN have no classifiable signal and are excluded. The aggregation denominators therefore reflect only clearly classified projects.
- **Missing investment data:** 606 rows have `TotalInvestment` as NaN (coerced from "Not Available" in the previous step). The `total_investment` and `resilient_investment` columns sum only non-NaN values (`skipna=True` is the pandas default), so investment-based shares may be based on fewer projects than count-based shares for the same country-year.

In [ ]:
groupby_cols = ['Country', 'Financial closure year']

# 1. Drop ambiguous rows
df_clean = df[df['is_resilient'].notna()].copy()
print(f'Rows after dropping ambiguous (is_resilient NaN): {len(df_clean)}')

# 2. Ensure TotalInvestment is numeric (idempotent if already converted)
df_clean['TotalInvestment'] = pd.to_numeric(df_clean['TotalInvestment'], errors='coerce')

# 3. Aggregate
totals = df_clean.groupby(groupby_cols).agg(
    total_projects=('Country', 'count'),
    total_investment=('TotalInvestment', 'sum'),
).reset_index()

resilient = (
    df_clean[df_clean['is_resilient'] == 1]
    .groupby(groupby_cols)
    .agg(
        resilient_projects=('Country', 'count'),
        resilient_investment=('TotalInvestment', 'sum'),
    )
    .reset_index()
)

panel = totals.merge(resilient, on=groupby_cols, how='left')
panel[['resilient_projects', 'resilient_investment']] = (
    panel[['resilient_projects', 'resilient_investment']].fillna(0)
)
panel['resilient_share_count'] = panel['resilient_projects'] / panel['total_projects']
panel['resilient_share_value'] = panel['resilient_investment'] / panel['total_investment']

panel = panel.sort_values(groupby_cols).reset_index(drop=True)

# 4. Print summary
print('\nPanel shape:', panel.shape)
print(f'Unique countries: {panel["Country"].nunique()}')
print(f'Unique years:     {panel["Financial closure year"].nunique()}')
print(f'Year range:       {int(panel["Financial closure year"].min())} – {int(panel["Financial closure year"].max())}')

print('\nFirst 10 rows:')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)
print(panel.head(10).to_string(index=False))

print('\nDistribution of total_projects per country-year:')
print(panel['total_projects'].describe())

single = (panel['total_projects'] == 1).sum()
print(f'\nCountry-years with only 1 project: {single} ({single/len(panel)*100:.1f}%)')

## 5. Panel Comparison: Electricity-Only vs Full-Sector Resilient

We compare the new country-year panel to the electricity-only panel built in `01_ppi_cleaning.ipynb`. The key question is whether expanding to all sectors meaningfully improves coverage — more country-years, more countries, a longer time horizon — without changing the fundamental sparsity problem (many country-years with very few projects).

More country-years and countries increases statistical power and reduces selection bias from focusing on electricity-active countries only. A longer time horizon (2000–2024 vs 2010–2024) also allows us to capture the pre-2010 baseline period in trend analyses.

In [ ]:
comparison = {
    'Metric': [
        'Country-years',
        'Countries',
        'Years covered',
        'Year range',
        'Avg projects per country-year',
        'Country-years with 1 project',
    ],
    'OLD (electricity-only)': [
        556,
        100,
        15,
        '2010–2024',
        4.7,
        '237 (42.6%)',
    ],
    'NEW (resilient full-sector)': [
        1163,
        129,
        25,
        '2000–2024',
        6.5,
        '495 (42.6%)',
    ],
}

print(f"{'Metric':<38} {'OLD (electricity-only)':<26} {'NEW (resilient full-sector)'}")
print('-' * 90)
for m, old, new in zip(comparison['Metric'],
                       comparison['OLD (electricity-only)'],
                       comparison['NEW (resilient full-sector)']):
    print(f'{m:<38} {str(old):<26} {new}')

# The sparsity rate is similar (42.6% in both) but the panel is
# larger overall — more countries, more years, more total observations.
# The weighting approach (weighting by total_projects in regression)
# remains necessary to handle single-project country-years.

## 6. Save Output Files

We save two files to `data/clean/`:

- **`ppi_resilient_share.csv`** — the country-year aggregated panel with resilient share metrics. This is the primary outcome variable for the main regression analysis.
- **`ppi_resilient_project_level.csv`** — the project-level dataframe with `is_resilient` attached, restricted to non-ambiguous rows (NaN rows dropped). This preserves the full project detail for robustness checks, subgroup analyses, or alternative aggregations.

In [ ]:
import os

panel_path   = '../../data/clean/ppi_resilient_share.csv'
project_path = '../../data/clean/ppi_resilient_project_level.csv'

# Project-level: non-ambiguous rows only (is_resilient not NaN)
df_project = df[df['is_resilient'].notna()].copy()

# Save
panel.to_csv(panel_path, index=False)
df_project.to_csv(project_path, index=False)

# Confirm
for path, label, obj in [
    (panel_path,   'ppi_resilient_share (country-year panel)', panel),
    (project_path, 'ppi_resilient_project_level',              df_project),
]:
    size_kb = os.path.getsize(path) / 1024
    print(f'--- {label} ---')
    print(f'  Path:    {path}')
    print(f'  Shape:   {obj.shape}')
    print(f'  Columns: {obj.columns.tolist()}')
    print(f'  Size:    {size_kb:.1f} KB')
    print()